# Combining the NLA Amplitude and Matter Power Spectrum

`NLA.ipynb` derived and tested each intrinsic-alignment factor separately. Here we
take those definitions as established and focus on the next computational step:
combining the established factors into the two-dimensional model-dependent factor
and signed IA amplitude, then applying that amplitude to the nonlinear matter power
spectrum.

## The path through the notebook

By the end, you should be able to:

- separate cosmology-dependent and model-dependent amplitude components;
- use NumPy broadcasting to construct a surface on a $(z,k)$ grid;
- verify the sign, shape, and factorization of the total IA amplitude;
- construct $P_{\delta I}$ and $P_{II}$ from $P_{\rm nl}$;
- identify several possible targets for ML-based compression.

This is a completed lecture notebook. The practical parameter-sampling exercise is
developed in `SAMPLE.ipynb`.

---

## Set up an independent calculation

This notebook imports the tested amplitude functions from `Alignment.py`. It can be run
from a fresh kernel and does not rely on variables left behind by `NLA.ipynb`.

In [ ]:
import json
import sys
from pathlib import Path

import numpy
import pyccl
from matplotlib import pyplot

In [ ]:
pyplot.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 20,
    "text.usetex": True,
})

colour_list = ["red", "orange", "black", "blue", "purple"]

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this notebook from within the IAFlowCloud repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
code_path = PROJECT_ROOT / "Code"
if str(code_path) not in sys.path:
    sys.path.insert(0, str(code_path))
base_path = PROJECT_ROOT
data_path = base_path / "Data" / "General"
figure_path = base_path / "Figure" / "NLA" / "Power"
figure_path.mkdir(parents=True, exist_ok=True)
print("Project directory:", base_path)
print("Code directory:   ", code_path)
print("Data directory:   ", data_path)
print("Figure directory: ", figure_path)


### Fix a fiducial cosmology and evaluation grid

The purpose of this lecture is amplitude recombination, so we use one Planck-like
cosmology and one fixed $(z,k)$ grid. `SAMPLE.ipynb` keeps cosmology outside its
sampling exercise so that nuisance-model variation can be studied independently.

In [ ]:
with open(data_path / "Planck.json", "r", encoding="utf-8") as file:
    parameter = json.load(file)

print(parameter)

In [ ]:
cosmology = pyccl.cosmology.Cosmology(
    h=parameter["H"],
    w0=parameter["W0"],
    wa=parameter["WA"],
    A_s=parameter["AS"],
    n_s=parameter["NS"],
    m_nu=parameter["MNU"],
    T_CMB=parameter["TCMB"],
    Omega_k=parameter["OMEGAK"],
    Omega_c=parameter["OMEGAC"],
    Omega_b=parameter["OMEGAB"],
    mass_split="normal",
    transfer_function="boltzmann_camb",
    extra_parameters={
        "camb": {
            "kmax": 100,
            "lmax": 5000,
            "halofit_version": "mead2020_feedback",
            "HMCode_logT_AGN": 7.8,
        }
    },
)

In [ ]:
z1 = 0.0
z2 = 3.0
z_size = 30
z = numpy.linspace(z1, z2, z_size + 1)

logk1 = -3.0
logk2 = +1.0
logk_size = 100
k = numpy.logspace(logk1, logk2, logk_size + 1)

r_star = pivot_redshift_ratio(z, z_star=Z_STAR)
z_plot_list = [0.0, 0.5, 1.0, 2.0, 3.0]
z_plot_indices = [
    numpy.flatnonzero(numpy.isclose(z, z_value))[0]
    for z_value in z_plot_list
]

fiducial_model = NLAModel(
    A0=1.0,
    eta=0.5,
    xi=0.0,
    s=2.0,
    z_q=1.0,
    q=1.0,
    n_star=2.0,
    k_t_star=0.5,
    alpha=0.3,
    m=2.0,
    gamma_t=0.4,
    gamma_n=0.2,
    gamma_alpha=0.0,
    gamma_m=0.0,
    constant=C0,
    z_star=Z_STAR,
)

print("z shape:           ", z.shape)
print("k shape:           ", k.shape)
print("Plotted redshifts: ", z_plot_list)
print("r_star:            ", r_star)
print("fiducial parameters:", fiducial_model.to_dict())

---

## Recap the amplitude decomposition

The previous notebook established

$$
\boxed{
\mathcal A_{\rm IA}(k,z)
=-A_0\,\mathcal A_\Omega(z)\,\mathcal A_\Theta(k,z)
}
$$

with

$$
\mathcal A_\Omega(z)=C_0\frac{\Omega_m}{D(z)}
$$

and

$$
\boxed{
\mathcal A_\Theta(k,z)
=R_z(z;\eta,z_\ast)\,R_L(z;\xi,s,z_q,z_\ast)\,S(k,z),
\qquad
R_z(z)=r_\ast^\eta(z).
}
$$

The shared coordinate $r_\ast=(1+z)/(1+z_\ast)$ also drives the optional evolution
of $k_t(z)$, $n(z)$, $\alpha(z)$, and $m(z)$. Here $\gamma_\alpha=\gamma_m=0$
selects the constant-$\alpha$, constant-$m$ fiducial case. The derivation and
parameter plots for $S(k,z)$ are already
given in `NLA.ipynb`, so we do not display that factor separately here. We use the
established functions through an immutable `NLAModel` instance and focus on array
recombination and power spectra.

---

## Broadcast one-dimensional factors onto a two-dimensional grid

$\mathcal A_\Omega$ depends only on redshift and has shape $(N_z,)$. The
model-dependent component depends on both redshift and scale and has shape
$(N_z,N_k)$. NumPy inserts a singleton scale axis,

$$
\mathcal A_\Omega(z)
\longrightarrow
\mathcal A_\Omega(z)[:,{\tt None}],
$$

so that

$$
(N_z,1)\times(N_z,N_k)
\longrightarrow
(N_z,N_k).
$$

The positive $\mathcal A_\Theta$ is the fixed-cosmology compression target. The
single scalar $A_0$ remains outside this surface and is applied exactly during
recombination.

In [ ]:
components = fiducial_model.amplitude_components(cosmology, z, k)

A_omega = components["A_omega"]
A_theta = components["A_theta"]
A_IA = components["A_IA"]

print("A_omega shape:   ", A_omega.shape)
print("A_theta shape:   ", A_theta.shape)
print("A_IA shape:      ", A_IA.shape)

### Validate the factorization

Before plotting, check the expected shapes and verify

$$
\mathcal A_{\rm IA}
=
-A_0\,\mathcal A_\Omega[:,{\tt None}]
\mathcal A_\Theta.
$$

These inexpensive checks catch transposed grids, missing singleton axes, and sign
mistakes before they propagate into the power spectra.

In [ ]:
expected_shape = (len(z), len(k))
recombined_A_IA = -fiducial_model.A0 * A_omega[:, None] * A_theta

assert A_omega.shape == (len(z),)
assert A_theta.shape == expected_shape
assert A_IA.shape == expected_shape
assert numpy.allclose(A_IA, recombined_A_IA)
assert numpy.all(numpy.isfinite(A_theta))
assert numpy.all(numpy.isfinite(A_IA))
assert numpy.all(A_theta > 0.0)
if fiducial_model.A0 > 0.0:
    assert numpy.all(A_IA < 0.0)
elif fiducial_model.A0 < 0.0:
    assert numpy.all(A_IA > 0.0)
else:
    assert numpy.all(A_IA == 0.0)

pivot_index = numpy.flatnonzero(numpy.isclose(z, Z_STAR))[0]
assert numpy.isclose(r_star[pivot_index], 1.0)

maximum_recombination_error = numpy.max(numpy.abs(A_IA - recombined_A_IA))
print("Maximum recombination error:", maximum_recombination_error)
print("A_theta is positive and independent of A0.")
print("The sign of A_IA is controlled by -A0.")

### Inspect the combined amplitude surfaces

We show the model-dependent factor and signed total amplitude in separate figures.
The internal scale factor is not repeated. We display $-\mathcal A_{\rm IA}$
because the fiducial signed amplitude is negative.

In [ ]:
figure, plot = pyplot.subplots(figsize=(10, 8))

mesh = plot.pcolormesh(
    numpy.log10(k),
    z,
    A_theta,
    shading="auto",
    cmap="plasma",
    rasterized=True,
)

plot.set_xlim(logk1, logk2)
plot.set_ylim(z1, z2)
plot.set_xlabel(r"$\log_{10}(k/{\rm Mpc}^{-1})$")
plot.set_ylabel(r"$z$")
plot.set_title(r"Model-dependent factor $\mathcal A_\Theta(k,z)$")
figure.colorbar(mesh, ax=plot, label=r"$\mathcal A_\Theta(k,z)$")

figure.tight_layout()
figure.savefig(figure_path / "Model_Factor_Surface.pdf", dpi=512, bbox_inches="tight")

In [ ]:
figure, plot = pyplot.subplots(figsize=(10, 8))

mesh = plot.pcolormesh(
    numpy.log10(k),
    z,
    -A_IA,
    shading="auto",
    cmap="plasma",
    rasterized=True,
)

plot.set_xlim(logk1, logk2)
plot.set_ylim(z1, z2)
plot.set_xlabel(r"$\log_{10}(k/{\rm Mpc}^{-1})$")
plot.set_ylabel(r"$z$")
plot.set_title(r"Signed IA amplitude $-\mathcal A_{\rm IA}(k,z)$")
figure.colorbar(mesh, ax=plot, label=r"$-\mathcal A_{\rm IA}(k,z)$")

figure.tight_layout()
figure.savefig(figure_path / "IA_Amplitude_Surface.pdf", dpi=512, bbox_inches="tight")

---

## Apply the amplitude to the nonlinear matter spectrum

For NLA,

$$
\boxed{
P_{\delta I}(k,z)
=
\mathcal A_{\rm IA}(k,z)P_{\rm nl}(k,z)
}
$$

and

$$
\boxed{
P_{II}(k,z)
=
\mathcal A_{\rm IA}^2(k,z)P_{\rm nl}(k,z).
}
$$

The cross-spectrum retains the sign of $\mathcal A_{\rm IA}$, while the auto-spectrum
is non-negative.

In [ ]:
P_m = numpy.vstack([
    pyccl.nonlin_matter_power(
        cosmology,
        k,
        1.0 / (1.0 + z_value),
    )
    for z_value in z
])

P_deltaI = A_IA * P_m
P_II = A_IA**2 * P_m

assert P_m.shape == A_IA.shape
assert numpy.all(numpy.isfinite(P_m))
assert numpy.all(numpy.isfinite(P_deltaI))
assert numpy.all(numpy.isfinite(P_II))
assert numpy.all(P_m > 0.0)
assert numpy.allclose(P_deltaI / P_m, A_IA)
assert numpy.allclose(P_II / P_m, A_IA**2)
assert numpy.all(P_deltaI < 0.0)
assert numpy.all(P_II >= 0.0)

print("P_m shape:      ", P_m.shape)
print("P_deltaI shape: ", P_deltaI.shape)
print("P_II shape:     ", P_II.shape)

### Inspect the resulting power spectra

Each spectrum is shown in a separate figure. A logarithmic axis cannot display a
negative cross-spectrum, so we plot $-P_{\delta I}$ and label the sign explicitly.
The signed values remain unchanged in memory.

In [ ]:
figure, plot = pyplot.subplots(figsize=(10, 8))

for index, z_value, colour in zip(
    z_plot_indices,
    z_plot_list,
    colour_list,
):
    plot.plot(
        k,
        P_m[index],
        color=colour,
        linewidth=2,
        label=rf"$z={z_value:.1f}$",
        rasterized=True,
    )

plot.set_xscale("log")
plot.set_yscale("log")
plot.set_xlim(10 ** logk1, 10 ** logk2)
plot.set_xlabel(r"$k$ [Mpc$^{-1}$]")
plot.set_ylabel(r"$P_{\rm nl}(k,z)$ [Mpc$^3$]")
plot.set_title("Nonlinear matter spectrum")
plot.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))

figure.tight_layout()
figure.savefig(figure_path / "Matter_Power_Spectrum.pdf", dpi=512, bbox_inches="tight")

In [ ]:
figure, plot = pyplot.subplots(figsize=(10, 8))

for index, z_value, colour in zip(
    z_plot_indices,
    z_plot_list,
    colour_list,
):
    plot.plot(
        k,
        -P_deltaI[index],
        color=colour,
        linewidth=2,
        label=rf"$z={z_value:.1f}$",
        rasterized=True,
    )

plot.set_xscale("log")
plot.set_yscale("log")
plot.set_xlim(10 ** logk1, 10 ** logk2)
plot.set_xlabel(r"$k$ [Mpc$^{-1}$]")
plot.set_ylabel(r"$-P_{\delta I}(k,z)$ [Mpc$^3$]")
plot.set_title("Matter--intrinsic spectrum")
plot.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))

figure.tight_layout()
figure.savefig(figure_path / "Matter_Intrinsic_Power_Spectrum.pdf", dpi=512, bbox_inches="tight")

In [ ]:
figure, plot = pyplot.subplots(figsize=(10, 8))

for index, z_value, colour in zip(
    z_plot_indices,
    z_plot_list,
    colour_list,
):
    plot.plot(
        k,
        P_II[index],
        color=colour,
        linewidth=2,
        label=rf"$z={z_value:.1f}$",
        rasterized=True,
    )

plot.set_xscale("log")
plot.set_yscale("log")
plot.set_xlim(10 ** logk1, 10 ** logk2)
plot.set_xlabel(r"$k$ [Mpc$^{-1}$]")
plot.set_ylabel(r"$P_{II}(k,z)$ [Mpc$^3$]")
plot.set_title("Intrinsic--intrinsic spectrum")
plot.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))

figure.tight_layout()
figure.savefig(figure_path / "Intrinsic_Intrinsic_Power_Spectrum.pdf", dpi=512, bbox_inches="tight")

---

## Choose the object to compress

The factorization identifies the focused ML problem used in this project:

### Complete model compression at fixed cosmology

Compress

$$
\mathcal A_\Theta(k,z\mid\boldsymbol\Theta).
$$

This positive surface contains every model-dependent shape factor while excluding the
global normalization $A_0$ and cosmological factor $\mathcal A_\Omega$. We therefore
compress $\log_{10}\mathcal A_\Theta$ and store $A_0$ as one separate scalar. After
decoding, the full signed amplitude is reconstructed exactly as
$-A_0\mathcal A_\Omega\widehat{\mathcal A}_\Theta$.

### Lecture checkpoints

Before moving on, make sure you can explain:

- why $\mathcal A_\Omega$ has shape $(N_z,)$ while
  $\mathcal A_\Theta$ has shape $(N_z,N_k)$;
- why $R_z=r_\ast^\eta$ remains an explicit factor even though it is simple;
- why a singleton axis is needed during recombination;
- how the established $S(k,z)$ factor propagates through $\mathcal A_\Theta$
  without being recomputed or plotted separately;
- why the fiducial $\mathcal A_{\rm IA}$ and $P_{\delta I}$ are negative while
  $P_{II}$ is non-negative;
- why $A_0$ remains an inference parameter but is not part of the compressed surface;
- why factorized storage is useful for checking an ML compression.

---

## Continue to parameter sampling

`SAMPLE.ipynb` provides a complete nuisance-only sampling example and a student
workspace for comparing candidate nuisance-parameter prior ranges while cosmology is
kept outside the sampled dataset.

## Final consistency checks


In [ ]:
assert (data_path / "Planck.json").is_file()
assert figure_path.is_dir()
assert numpy.all(numpy.isfinite(P_m))
assert numpy.all(numpy.isfinite(P_deltaI))
assert numpy.all(numpy.isfinite(P_II))
print("Power notebook consistency checks passed.")
